# OpenAI API 사용량 확인
**SK하이닉스 Autonomous R&D — Day 3**

> **중요:** OpenAI는 **남은 토큰 개수**를 일반 API 키로 조회하는 공식 API를 제공하지 않습니다.
>
> - 과금은 보통 사용한 만큼 USD로 청구됩니다.
> - 잔여 크레딧은 [Billing 페이지](https://platform.openai.com/settings/organization/billing)에서 확인하세요.

이 노트북에서 할 수 있는 것:
1. API 키가 정상인지 간단히 확인
2. 테스트 호출 1회에서 사용한 토큰 출력
3. (선택) 조직 Admin API 키가 있으면 최근 7일 토큰 사용량 합계 출력

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [1]:
!pip install openai python-dotenv httpx

---
## 1. 환경 설정 (.env)

In [2]:
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional

import httpx
from dotenv import load_dotenv
from openai import OpenAI

BASE_DIR = Path.cwd()
for env_path in (BASE_DIR / '.env', BASE_DIR.parent / '.env'):
    if env_path.exists():
        load_dotenv(env_path)
        print('✅ .env 로드:', env_path.resolve())
        break
else:
    load_dotenv()
    print('⚠️ .env 파일을 찾지 못했습니다. 환경변수를 직접 사용합니다.')

MODEL = 'gpt-4o-mini'

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ API 클라이언트 준비 완료')

✅ .env 로드: C:\Users\Admin\Desktop\AI Autonomous\cursor\.env
✅ API 클라이언트 준비 완료


---
## 2. 테스트 호출 · 이번 요청 토큰 사용량

`response.usage`로 **이번 호출**에서 쓴 토큰을 확인합니다. (잔여 토큰이 아님)

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    max_tokens=20,
    messages=[
        {'role': 'user', 'content': 'Reply with only the word OK.'},
    ],
)

usage = response.usage
print(f'모델: {MODEL}')
print(f'답변: {response.choices[0].message.content}')
print(f'입력 토큰(prompt):     {usage.prompt_tokens}')
print(f'출력 토큰(completion): {usage.completion_tokens}')
print(f'총 토큰(total):        {usage.total_tokens}')
print()
print("※ 위 숫자는 '이번 호출'에서 쓴 토큰입니다. 잔여 토큰이 아닙니다.")

모델: gpt-4o-mini
답변: OK
입력 토큰(prompt):     14
출력 토큰(completion): 1
총 토큰(total):        15

※ 위 숫자는 '이번 호출'에서 쓴 토큰입니다. 잔여 토큰이 아닙니다.


---
## 3. (선택) 최근 7일 조직 토큰 사용량

조직 **Admin API 키**가 있을 때만 동작합니다.

- 발급: [Admin keys](https://platform.openai.com/settings/organization/admin-keys)
- `.env`에 `OPENAI_ADMIN_API_KEY=sk-admin-...` 추가

In [4]:
def fetch_recent_usage_total(admin_api_key: str, days: int = 7) -> Optional[int]:
    """조직 Admin API 키로 최근 N일 토큰 사용량 합계를 조회한다."""
    start_time = int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp())
    url = 'https://api.openai.com/v1/organization/usage/completions'
    headers = {
        'Authorization': f'Bearer {admin_api_key}',
        'Content-Type': 'application/json',
    }
    params = {
        'start_time': start_time,
        'bucket_width': '1d',
        'limit': days,
    }

    response = httpx.get(url, headers=headers, params=params, timeout=30.0)
    if response.status_code != 200:
        print(f'조회 실패 (HTTP {response.status_code})')
        print(response.text)
        return None

    data = response.json()
    total_tokens = 0
    for bucket in data.get('data', []):
        for result in bucket.get('results', []):
            total_tokens += int(result.get('input_tokens', 0))
            total_tokens += int(result.get('output_tokens', 0))
    return total_tokens


admin_api_key = os.getenv('OPENAI_ADMIN_API_KEY')

if admin_api_key:
    total = fetch_recent_usage_total(admin_api_key, days=7)
    if total is not None:
        print(f'최근 7일 사용 토큰 합계: {total:,}')
        print("※ 이것도 '남은 토큰'이 아니라 '사용한 토큰'입니다.")
else:
    print('조직 Admin API 키가 없어 최근 사용량 조회를 건너뜁니다.')
    print('필요하면 .env에 OPENAI_ADMIN_API_KEY=sk-admin-... 를 추가하세요.')

조직 Admin API 키가 없어 최근 사용량 조회를 건너뜁니다.
필요하면 .env에 OPENAI_ADMIN_API_KEY=sk-admin-... 를 추가하세요.
